# HW 2: Discrete Heterogeneity

Part 1: Re-estimate the multinomial logit brand choice model with the last brand purchased variable on the Yogurt data, but using a 2 type discrete heterogeneity distribution.  For now put the heterogeneity only on the intercepts. The last assignment had you form an individual specific likelihood function that took an NT by 1 vector and collapsed it to N by 1. For this assignment, first form a likelihood function that is NT by M. Then take the products across T within the first dimension to collapse it into an N by M matrix with the likelihood for each individual if they were each type. Then, take the weighted average of the types over the 2nd dimension to form the individual specific likelihood function which is N by 1.

Part 2: Now try allowing all of the parameters to be type specific (still use 2 types).  Also, try allowing for 4 types but restricting types to vary only by intercepts.

Part 3: Numerically solve for cross price elasticities in the homogenous multinomial logit model and the two type finite support heterogeneity model.  To do so, simulate a small change in the price of brand 1.  Also try it by simulating a small price change for brand 2. Compare elasticities between the homogenous and heterogenous models.

## Results from STATA

```text

```

## Python implementation

In [1]:
# Imports
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import logsumexp
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess

In [2]:
# Import data
df = pd.read_csv("../data/YogurtLong.txt", sep="\t")
print(df.head())

   hh  qty        exp  nopurch  b1  b2  b3  b4    p1    p2    p3    p4  f1  \
0   1    2  40.900002        0   0   0   0   1  0.11  0.08  0.06  0.08   0   
1   1    0   8.830000        1   0   0   0   0  0.11  0.09  0.05  0.08   0   
2   1    0   3.880000        1   0   0   0   0  0.13  0.09  0.05  0.08   0   
3   1    0   0.780000        1   0   0   0   0  0.13  0.09  0.05  0.08   0   
4   1    0  39.240002        1   0   0   0   0  0.13  0.09  0.05  0.08   0   

   f2  f3  f4  tripnum  
0   0   0   0        1  
1   0   0   0        2  
2   0   0   0        3  
3   0   0   0        4  
4   0   0   0        5  


In [3]:
# Convert df to long (each row is a hh-trip-brand choice occasion)
df_long = (
    pd.wide_to_long(
        df,
        stubnames=["b", "p", "f"],
        i=["hh", "tripnum"],
        j="brand",
        suffix=r"\d+"
    )
    .reset_index()
    .sort_values(["hh", "tripnum", "brand"])
)
df_long = pd.get_dummies(df_long, columns=["brand"], prefix="brand", dtype=int)
df_long.head()

,hh,tripnum,exp,nopurch,qty,b,p,f,brand_1,brand_2,brand_3,brand_4
0,1,1,40.900002,0,2,0,0.11,0,1,0,0,0
1,1,1,40.900002,0,2,0,0.08,0,0,1,0,0
2,1,1,40.900002,0,2,0,0.06,0,0,0,1,0
3,1,1,40.900002,0,2,1,0.08,0,0,0,0,1
4,1,2,8.830000,1,0,0,0.11,0,1,0,0,0


### I. 2-type discrete heterogeneity on intercepts



In [4]:
def loglik(beta, X, Y, hh_ids, n_alts):
    """
    Computes negative log-likelihood for the dataset, assuming a conditional logit model with an outside option.

    Parameters
    ----------
    beta : array
        Coefficients to be estimated.
    X : array
        Matrix of covariates
    Y : array
        Vector of choices
    hh_ids : array
        Household ID corresponding to each row of X and Y.
    n_alts : int
        Number of alternatives.

    Returns
    -------
    float
        Negative log-likelihood.
    """
    X = np.asarray(X)
    Y = np.asarray(Y).reshape(-1)
    hh_ids = np.asarray(hh_ids).reshape(-1)
    beta = np.asarray(beta).reshape(-1)

    n, k = X.shape
    n_choices = n // n_alts                         # number of hh-trip choice occasions

    v_ij = X @ beta                                 # utility for each hh-trip-brand
    V = v_ij.reshape(n_choices, n_alts, order="C")  # reshape to (choice occasion, alternative)
    chosen_v = np.multiply(Y, v_ij).\
        reshape(n_choices, n_alts).sum(axis=1)      # chosen utility per choice occasion
    V_with_outside = np.hstack([np.zeros((n_choices, 1)), V])   # add outside option (zero utility)
    log_denom = logsumexp(V_with_outside, axis=1)

    loglik_trip = chosen_v - log_denom                          # trip-level log-likelihoods
    _, hh_index = np.unique(hh_ids, return_inverse=True)        # group by household
    loglik_hh = np.bincount(hh_index, weights=loglik_trip)

    return -np.sum(loglik_hh)


def score_matrix(beta, X, Y, n_alts):
    """
    Returns choice-occasion score vectors s_t(beta) = d log L_t / d beta.
    Shape: (n_choices, k).
    """
    X = np.asarray(X)
    Y = np.asarray(Y).reshape(-1)
    beta = np.asarray(beta).reshape(-1)

    n, k = X.shape
    n_choices = n // n_alts

    # Reshape long format into (choice occasion, alternative, regressor)
    X3 = X.reshape(n_choices, n_alts, k, order="C")
    Y2 = Y.reshape(n_choices, n_alts, order="C")

    # Softmax probabilities including outside option with normalized utility 0
    V = X3 @ beta                                      # (n_choices, n_alts)
    V_with_outside = np.hstack([np.zeros((n_choices, 1)), V])
    P_with_outside = np.exp(V_with_outside - logsumexp(V_with_outside, axis=1, keepdims=True))
    P = P_with_outside[:, 1:]                          # drop outside option prob

    # Score per choice occasion: sum_j (y_tj - p_tj) x_tj
    S = np.einsum("ta,tak->tk", (Y2 - P), X3)
    return S


def mle_estimator(df, 
                  choice_var,
                  covariate_vars,
                  individual_ids,
                  n_alts,
                  beta_init=None,
                  ci_alpha=0.05,
                  robust_se=False,
                  method='BFGS'):
    """
    Estimates the multinomial logit model using maximum likelihood estimation.
    
    Parameters    
    ----------
    df : DataFrame
        The dataset to be used for estimation.
    choice_var : str
        The column name of the choice variable.
    covariate_vars : list
        A list of column names for the covariates.
    individual_ids : array
        An array of individual IDs.
    n_alts : int
        Number of alternatives.
    beta_init : array, optional
        Initial guess for coefficients. If None, defaults to zeros.
    ci_alpha : float
        Significance level for confidence intervals.
    robust_se : bool
        Whether to compute robust standard errors.
    method : str
        Optimization method.

    Returns    
    -------
    Dictionary containing:
        - optimized log-likelihood 
        - coefficient estimates
        - standard errors
        - confidence intervals for coefficients
    """
    # Prepare data
    Y = df[[choice_var]].to_numpy()
    X = df[covariate_vars].to_numpy()

    if beta_init is None:
        beta_init = np.zeros(X.shape[1])

    result = minimize(
        loglik, 
        beta_init, 
        args=(X, Y, individual_ids, n_alts),
        method=method,
    )

    opt_ll = -result.fun
    opt_beta = result.x

    # Standard errors and confidence intervals (using Hessian, under correct specification)
    H = approx_hess(opt_beta, loglik, args=(X, Y, individual_ids, n_alts))  # numerical Hessian at opt_beta
    H_inv = np.linalg.inv(H)
    se_beta = np.sqrt(np.diag(H_inv))
    z_score = norm.ppf(1 - ci_alpha / 2)
    ci = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_beta, se_beta)]

    output = {
        'opt_ll': opt_ll,
        'opt_beta': opt_beta,
        'se_beta': se_beta,
        'ci': ci,
    }

    if robust_se:
        # J = E[s_t s_t'] estimated by sample average of outer products of scores.
        S = score_matrix(opt_beta, X, Y, n_alts)       # (n_choices, k)
        n_choices = S.shape[0]
        J = S.T @ S

        # Sandwich variance: H^{-1} J H^{-1}
        V_robust = H_inv @ J @ H_inv
        se_beta_r = np.sqrt(np.diag(V_robust))
        ci_r = [(coef - z_score * se, coef + z_score * se) for coef, se in zip(opt_beta, se_beta_r)]

        output['se_beta'] = se_beta_r
        output['ci'] = ci_r
    
    # Print coefficient estimates with aligned decimals and grouped CI header
    widths = (12, 10, 10, 10, 10)

    header_top = f"{'Coefficient':^{widths[0]}} | {'Estimate':^{widths[1]}} | {'Std. Err.':^{widths[2]}} | {'Confidence Interval':^{widths[3] + widths[4] + 3}}"
    divider = "-+-".join("-" * w for w in widths)
    print(header_top)
    print(divider)

    for i, (name, coef) in enumerate(zip(covariate_vars, opt_beta)):
        row = " | ".join([
            f"{name:<{widths[0]}}",
            f"{coef:>{widths[1]}.6f}",
            f"{output['se_beta'][i]:>{widths[2]}.7f}",
            f"{output['ci'][i][0]:>{widths[3]}.5f}",
            f"{output['ci'][i][1]:>{widths[4]}.5f}"
        ])
        print(row)

    return output

In [5]:
df_long["prev_purch"] = df_long.groupby("hh")["b"].shift(4).fillna(0).astype(int)
df_long.head()

hh_ids = df[['hh']].to_numpy()
results = mle_estimator(df_long, 
                        choice_var = 'b', 
                        covariate_vars = ['brand_1', 'brand_2', 'brand_3', 'brand_4', 'f', 'p', 'prev_purch'],
                        individual_ids = hh_ids, 
                        n_alts = 4, 
                        robust_se = False)

Coefficient  |  Estimate  | Std. Err.  |   Confidence Interval  
-------------+------------+------------+------------+-----------
brand_1      |   0.562170 |  0.1680839 |    0.23273 |    0.89161
brand_2      |  -0.144805 |  0.1342376 |   -0.40791 |    0.11830
brand_3      |  -3.361725 |  0.1444082 |   -3.64476 |   -3.07869
brand_4      |  -0.632828 |  0.1340291 |   -0.89552 |   -0.37014
f            |   0.353774 |  0.0968841 |    0.16388 |    0.54366
p            | -35.140550 |  1.5778514 |  -38.23308 |  -32.04802
prev_purch   |   3.182005 |  0.0511430 |    3.08177 |    3.28224
